In [ ]:
import sys
print(sys.executable)

In [ ]:
%load_ext autoreload
%autoreload 2

# Atari 视频帧 VQ-VAE 重建实验

**目标**：Atari rollout 帧打散为独立图像，训练帧级 VQ-VAE，单帧高精度重建。

**数据约定**：`(B, 3, 64, 64)`，帧级 shuffle，train / test 划分。

**待实现模块**（`Video_Ex/`）：
- `GetData.py` → `get_AtariFrame(batch_size)`
- `Model.py` → Transformer VQ-VAE
- `Train.py` → 训练 / 验证 / 可视化（接口对齐 `Image_Ex/Train.py`）

# 1. 导包 + 全局变量

In [ ]:
import GetData
import Model
import Train

import torch
import matplotlib.pyplot as plt
import os

In [ ]:
EPOCHS = 1
PRINT_EPOC = 1
BATCH_SIZE = 32
lr = 2e-4
wd = 0
device = "cuda:0" if torch.cuda.is_available() else "cpu"

mean = [0.0, 0.0, 0.0]
std  = [1.0, 1.0, 1.0]
IMAGE_SIZE = 64

IN_CHANNELS = 3
OUT_CHANNELS = 3
PATCH_SIZE = 4              # 64 → 16×16 潜空间
HIDDEN_DIM = 256
NUM_LAYERS = 6
NUM_HEADS = 8
MLP_RATIO = 4.0
NUM_EMBEDDINGS = 512
EMBEDDING_DIM = 64
DECAY = 0.99
COMMIT_COST = 0.25

# 2. 准备数据

**接口**：`GetData.get_AtariFrame(batch_size)` → `(train_loader, test_loader)`

```python
get_AtariFrame(batch_size)
# 内部：若 data/Atari/ 不存在则下载/采集
# 返回 batch: (images, label)，images.shape == (B, 3, 64, 64)
```

In [ ]:
# train_loader, test_loader = GetData.get_AtariFrame(BATCH_SIZE)

## 2.1 数据检查与可视化

In [ ]:
# first_batch = next(iter(train_loader))
# images = first_batch[0]
# print(images.shape)
# # 可视化若干帧 ...

# 3. 训练 VQ-VAE

**模型接口**：`Model.Model(...)`
- `forward(x)` → `(commit_loss, perplexity, x_recon)`
- `encode(x)` → `indices (B, 16, 16)`
- `decode(indices)` → `(B, 3, 64, 64)`

**训练接口**：`Train.train_pipeline(...)` · `Train.valid(...)` · `Train.draw(...)`

In [ ]:
# init_model = Model.Model(...)

In [ ]:
# perplexity_list, loss_list, recon_loss_list, commit_loss_list, best_model = Train.train_pipeline(
#     init_model, train_loader, EPOCHS, PRINT_EPOC, lr, wd, device, mean, std,
# )

## 3.1 训练曲线

In [ ]:
# plt.plot(loss_list, ...); plt.show()

In [ ]:
# plt.plot(perplexity_list, ...); plt.show()

## 3.2 保存 checkpoint

→ `checkpoints/best_vqvae_atari.pt`（供 `Atari_gene.ipynb` 加载）

In [ ]:
# torch.save({"model_state_dict": ..., "config": ...}, SAVE_PATH)

# 4. 测试集重建

`Train.valid(best_model, test_loader, device, mean, std, idx)`

In [ ]:
# Train.valid(best_model, test_loader, device, mean, std, idx=8)

# 5. （可选）encode / decode 抽查

In [ ]:
# indices = best_model.encode(x)
# x_dec = best_model.decode(indices)
# Train.draw(x, x_dec, n=10, mean=mean, std=std)